# Neo4j MCP Server + Haystack 3.0

Two building blocks, wired together by the **Model Context Protocol**: the official
**Neo4j MCP server** for graph access, and a **Haystack 3.0** `Agent` for orchestration.

Instead of writing Cypher retrievers into Haystack tools by hand — the approach in
[`neo4j_graphrag_haystack.ipynb`](./neo4j_graphrag_haystack.ipynb) — the MCP server exposes graph
access as a small set of named tools (`get-schema`, `read-cypher`, ...) over a standard protocol,
and Haystack's `mcp-haystack` integration connects to it directly. No retriever code, no index to
maintain — just a schema, a query tool, and an agent that reads before it writes Cypher.

| | |
| --- | --- |
| **Neo4j MCP server** | The protocol boundary — named tools, not a connection string |
| **Haystack 3.0** | Orchestration — `Agent`, `Pipeline`, tool selection |

Contents:

1. Start the Neo4j MCP server (HTTP transport, per-request auth)
2. Verify the tool list before involving an LLM
3. Connect it to a Haystack `Agent` via `MCPToolset`
4. Mix MCP tools with a custom Haystack `Tool`
5. A deterministic text-to-Cypher `Pipeline` — no agent freedom

Runs against the public Neo4j demo database. You need an OpenAI API key and nothing else.

---
## 1. Setup

In [14]:
%pip install -q "haystack-ai>=3.0" mcp-haystack neo4j-mcp-server neo4j python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import base64
import json
import logging
import os
import shutil
import socket
import subprocess
import sys
import time
import warnings
from getpass import getpass

import requests
from neo4j import GraphDatabase

from haystack import Pipeline, component
from haystack.dataclasses import ChatMessage
from haystack.tools import Tool
from haystack.components.agents import Agent
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator

from haystack_integrations.tools.mcp import MCPToolset, StreamableHttpServerInfo

# Keep the notebook output readable.
warnings.filterwarnings("ignore")
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

print("ready")

ready


---
## 2. Configuration

Two separate sets of credentials, same split as in the ADK notebook:

* **`OPENAI_API_KEY`** — the model that powers the agent.
* **Neo4j username / password** — the graph the agent queries, sent as a per-request header,
  never as an environment variable the MCP server process holds.

In [22]:
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
OPEN_AI_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = "gpt-5.4-mini"

# Public Neo4j demo database - no signup required.
NEO4J_URI = "neo4j+s://demo.neo4jlabs.com"
NEO4J_DATABASE = "companies"
NEO4J_USERNAME = "companies"
NEO4J_PASSWORD = "companies"

MCP_HOST = "127.0.0.1"
MCP_PORT = "8443"
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

print("Model:", OPENAI_MODEL)

Model: gpt-5.4-mini


---
## 3. Start the Neo4j MCP server

The server sits between the agent and the database: the agent calls named tools, never raw
Cypher over a connection string it holds itself.

We run it in **HTTP** mode rather than STDIO, so credentials travel as a per-request
`Authorization` header instead of environment variables fixed at process startup — the same
choice the [ADK notebook](../google-adk/google_adk.ipynb) makes, for the same reason: one
server process, many callers, each with their own credentials.

**In HTTP mode the server rejects `NEO4J_USERNAME` / `NEO4J_PASSWORD` at startup**, so they are
stripped from its environment before launch.

In [23]:
MCP_BINARY = shutil.which("neo4j-mcp") or shutil.which("neo4j-mcp-server")
MCP_COMMAND = [MCP_BINARY] if MCP_BINARY else [sys.executable, "-m", "neo4j_mcp_server"]

server_env = os.environ.copy()
server_env.update({
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_DATABASE": NEO4J_DATABASE,
    "NEO4J_TRANSPORT_MODE": "http",
    "NEO4J_MCP_HTTP_HOST": MCP_HOST,
    "NEO4J_MCP_HTTP_PORT": MCP_PORT,
    "NEO4J_READ_ONLY": "true",
    "NEO4J_TELEMETRY": "false",
    "NEO4J_SCHEMA_SAMPLE_SIZE": "100",
})

# HTTP mode is stateless and refuses startup credentials.
server_env.pop("NEO4J_USERNAME", None)
server_env.pop("NEO4J_PASSWORD", None)

process = subprocess.Popen(
    MCP_COMMAND, env=server_env,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)

# Wait for the port instead of sleeping a fixed amount.
deadline, listening = time.time() + 20, False
while time.time() < deadline and process.poll() is None:
    with socket.socket() as probe:
        probe.settimeout(0.5)
        if probe.connect_ex((MCP_HOST, int(MCP_PORT))) == 0:
            listening = True
            break
    time.sleep(0.25)

if listening:
    print("MCP server listening on", MCP_URL)
else:
    print("Server failed to start:\n", process.communicate(timeout=5)[1])

MCP server listening on http://127.0.0.1:8443/mcp


---
## 4. Authenticate and verify the tool list

`NEO4J_READ_ONLY=true` means `write-cypher` is never advertised — the model cannot call a tool
it was never shown. Confirming that here, with a direct JSON-RPC call, is easier to debug than
discovering it inside an agent transcript.

In [24]:
token = base64.b64encode(f"{NEO4J_USERNAME}:{NEO4J_PASSWORD}".encode()).decode()
AUTH_HEADERS = {"Authorization": f"Basic {token}"}

response = requests.post(
    MCP_URL,
    headers={**AUTH_HEADERS,
             "Content-Type": "application/json",
             "Accept": "application/json, text/event-stream"},
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
    timeout=30,
)
response.raise_for_status()

# Streamable HTTP may answer as JSON or as a server-sent event stream.
body = response.text
if body.startswith("event:") or "\ndata:" in body:
    body = next(l[5:] for l in body.splitlines() if l.startswith("data:"))

for tool in json.loads(body)["result"]["tools"]:
    print(f"  {tool['name']:<22} {tool.get('description', '').splitlines()[0]}")

  get-schema             
  list-gds-procedures    Use this tool to discover what graph science and analytics functions are available in the current Neo4j environment. It returns a structured list describing each function — what it does, how to use it, the inputs it needs, and what kind of results it produces. Do this before any reasoning, query generation, or analysis so you know what capabilities exist. Graph science and analytics functions help you with centrality, community detection, similarity, path finding, and identifying dependencies between nodes. The tool helps you understand the analytical capabilities of the system so that you can plan or compose the right graph science operations automatically. An empty response indicates that GDS is not installed and the user should be told to install it. Remember to use unique names for graph data science projections to avoid collisions and to drop them afterwards to save memory. You must always tell the user the function you will use.


---
## 5. Connect the graph to a Haystack `Agent`

`MCPToolset` connects to the server and loads its tools into Haystack in one step. Two
arguments worth setting:

* **`server_info`** — a `StreamableHttpServerInfo` carrying the URL and the same
  `Authorization` header verified above.
* **`tool_names`** — an allow-list on the Haystack side. It pairs with `NEO4J_READ_ONLY` for
  defence in depth, and shortens the tool schema sent with every prompt, which improves
  tool-selection accuracy — the same rationale as `tool_filter` in the ADK notebook.

In [25]:
mcp_tools = MCPToolset(
    server_info=StreamableHttpServerInfo(url=MCP_URL, headers=AUTH_HEADERS, timeout=30),
    tool_names=["get-schema", "read-cypher"],
)

graph_agent = Agent(
    chat_generator=OpenAIChatGenerator(model=OPENAI_MODEL),
    tools=mcp_tools,
    system_prompt="""You are a Neo4j graph database assistant.

1. Call `get-schema` before writing Cypher against an unfamiliar graph. Never guess at
   labels, relationship types, or property names.
2. Run read-only queries with `read-cypher`, always with a LIMIT clause.
3. If a query errors, read the error, fix the Cypher, and try again.
4. If the data is not in the graph, say so. Do not invent results.
""",
)

print("Agent ready.")

Agent ready.


---
## 6. Run it

One helper, reused for every example below — same shape as the `ask()` helper in
[`neo4j_graphrag_haystack.ipynb`](./neo4j_graphrag_haystack.ipynb), so tool calls are visible as
they happen.

In [26]:
def ask(agent, query, show_tools=True):
    print(f"USER: {query}")
    result = agent.run(messages=[ChatMessage.from_user(query)])

    for msg in result["messages"]:
        if show_tools and msg.tool_call:
            print(f"   -> {msg.tool_call.tool_name}({msg.tool_call.arguments})")
        elif show_tools and msg.tool_call_result:
            print(f"   <- {msg.tool_call_result.origin.tool_name}")

    answer = result["messages"][-1].text
    print(f"\nAGENT: {answer}\n")
    return answer


ask(graph_agent, "How many organizations are in the graph, and what node labels exist?")

USER: How many organizations are in the graph, and what node labels exist?
   -> get-schema({})
   <- get-schema
   -> read-cypher({'query': 'MATCH (o:Organization) RETURN count(o) AS organizationCount LIMIT 1'})
   <- read-cypher

AGENT: There are **46,088 Organization** nodes in the graph.

Node labels present:
- `Organization`
- `IndustryCategory`
- `Person`
- `Article`
- `Chunk`
- `City`
- `Country`
- `Fewshot`
- `_Bloom_Perspective_`
- `_Bloom_Scene_`



'There are **46,088 Organization** nodes in the graph.\n\nNode labels present:\n- `Organization`\n- `IndustryCategory`\n- `Person`\n- `Article`\n- `Chunk`\n- `City`\n- `Country`\n- `Fewshot`\n- `_Bloom_Perspective_`\n- `_Bloom_Scene_`'

---
## 7. Add your own tools alongside MCP tools

MCP gives the agent general-purpose graph access. A custom `Tool` gives it curated access: a
query you have already tuned, exposed under a name the model can reason about — both kinds sit
in the same `tools` list, exactly as `mcp_tools` and `FunctionTool(get_investments)` share one
list in the ADK notebook.

In [27]:
# One driver, reused. Creating a driver per call is a common performance mistake.
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))


def get_investments(company: str) -> str:
    """Return the companies that a given company has invested in.

    Args:
        company: Exact name of the Organization node, for example "Google".
    """
    records, _, _ = driver.execute_query(
        """
        MATCH (o:Organization)-[:HAS_INVESTOR]->(i)
        WHERE o.name = $company
        RETURN i.id AS id, i.name AS name, head(labels(i)) AS type
        LIMIT 50
        """,
        company=company,
        database_=NEO4J_DATABASE,
    )
    return json.dumps([r.data() for r in records], indent=2)


investments_tool = Tool(
    name="get_investments",
    description=get_investments.__doc__,
    function=get_investments,
    parameters={
        "type": "object",
        "properties": {"company": {"type": "string", "description": "Exact Organization name"}},
        "required": ["company"],
    },
)

investor_agent = Agent(
    chat_generator=OpenAIChatGenerator(model=OPENAI_MODEL),
    tools=[mcp_tools, investments_tool],
    system_prompt=(
        "You answer questions about a graph of company data. Use `get_investments` for "
        "questions about a company's investments. Use `get-schema` and `read-cypher` "
        "for everything else."
    ),
)

ask(investor_agent, "What has Google invested in?")

USER: What has Google invested in?
   -> get_investments({'company': 'Google'})
   <- get_investments

AGENT: Google has invested in:

- Ionic Security
- Avere Systems
- FlexiDAO
- Cloudflare
- Trifacta



'Google has invested in:\n\n- Ionic Security\n- Avere Systems\n- FlexiDAO\n- Cloudflare\n- Trifacta'

---
## 8. Cleanup

Leaving the MCP server running holds the port and makes the next run fail with a confusing
bind error.

In [32]:
try:
    process.terminate()
    process.wait(timeout=5)
except Exception:
    process.kill()
finally:
    driver.close()

print("Stopped.")

Stopped.


---
## Summary

**The MCP server is the protocol boundary, independent of the orchestrator.** It is the same
server, the same tools, the same `NEO4J_READ_ONLY` guarantee whether the caller is this
notebook's Haystack `Agent` or the ADK agent in the companion notebook — only the client-side
wiring changes.

**`MCPToolset` makes MCP tools first-class Haystack tools.** `tool_names` filters them, and they
sit in the same `tools` list as any `Tool` you write by hand — MCP for breadth, a custom `Tool`
for a query you have already tuned.

### Next steps

* [Neo4j MCP configuration](https://neo4j.com/docs/mcp/current/configuration/)
* [Neo4j MCP authentication](https://neo4j.com/docs/mcp/current/authentication/)
* [`mcp-haystack` reference](https://docs.haystack.deepset.ai/reference/integrations-mcp) —
  `MCPTool`, `MCPToolset`, `StdioServerInfo`, `SSEServerInfo`
* [Haystack Agents](https://docs.haystack.deepset.ai/docs/agents)
* [`neo4j_graphrag_haystack.ipynb`](./neo4j_graphrag_haystack.ipynb) — curated retrievers as
  Haystack tools, and the pipeline-vs-agent framing this notebook borrows from
* [`google_adk.ipynb`](../google-adk/google_adk.ipynb) — the same MCP server from ADK, plus
  persistent agent memory